## Общий класс узла дерева

In [2]:
class Node:
    def __init__(self, value, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right
        self.balance_factor = 0

In [3]:
def print_tree(root):
    if root is None:
        print('<empty tree>')
        return

    levels = []
    current = [root]
    while any(node is not None for node in current):
        levels.append(current)
        next_level = []
        for node in current:
            if node is None:
                next_level.extend([None, None])
            else:
                next_level.extend([node.left, node.right])
        current = next_level

    h = len(levels)
    max_len = max(len(str(node.value)) for level in levels for node in level if node is not None)
    step = max_len + 2
    width = (2 ** h - 1) * step

    for l, level in enumerate(levels):
        line = [' '] * width
        for i, node in enumerate(level):
            if node is None:
                continue
            col = ((2 * i + 1) * 2 ** (h - l - 1) - 1) * step
            value = str(node.value)
            start = col - (len(value) - 1) // 2
            for j, char in enumerate(value):
                if 0 <= start + j < width:
                    line[start + j] = char
        print(''.join(line).rstrip())

## 1. Полное бинарное дерево
Используем BFS: идём по уровням слева направо. Если встретили пустое место, после него уже не должно быть узлов. Сложность: **O(n)** по времени и **O(n)** по памяти.

In [4]:
from collections import deque


def is_complete(root):
    q = deque([root])
    found_empty = False

    while q:
        node = q.popleft()
        if node is None:
            found_empty = True
        else:
            if found_empty:
                return False
            q.append(node.left)
            q.append(node.right)

    return True


complete_tree = Node(1, Node(2, Node(4), Node(5)), Node(3, Node(6)))
incomplete_tree = Node(1, Node(2), Node(3, Node(6)))

assert is_complete(complete_tree) is True
assert is_complete(incomplete_tree) is False
assert is_complete(Node(1, None, Node(2))) is False
assert is_complete(None) is True
print('Полное дерево:')
print_tree(complete_tree)
print('Неполное дерево:')
print_tree(incomplete_tree)
print('Задача 1: OK')

Полное дерево:
         1
   2           3
4     5     6
Неполное дерево:
         1
   2           3
            6
Задача 1: OK


## 2. Объединение K отсортированных массивов
Кладём в мин-кучу первый элемент каждого непустого массива. После извлечения добавляем следующий элемент из того же массива. Если всего **N** элементов, сложность: **O(N log K)** по времени, **O(K)** дополнительной памяти (без результата).

In [5]:
import heapq


def merge_k_arrays(arrays):
    heap = []
    result = []

    for i in range(len(arrays)):
        if arrays[i]:
            heapq.heappush(heap, (arrays[i][0], i, 0))

    while heap:
        value, i, j = heapq.heappop(heap)
        result.append(value)

        if j + 1 < len(arrays[i]):
            heapq.heappush(heap, (arrays[i][j + 1], i, j + 1))

    return result


assert merge_k_arrays([[1, 4, 7], [2, 3, 9], [], [0, 5, 8]]) == [0, 1, 2, 3, 4, 5, 7, 8, 9]
assert merge_k_arrays([[], []]) == []
assert merge_k_arrays([[1, 1], [1, 2]]) == [1, 1, 1, 2]
print('Задача 2:', merge_k_arrays([[1, 4], [2, 5], [3, 6]]))

Задача 2: [1, 2, 3, 4, 5, 6]


## 3. K-й наименьший и наибольший элемент
Для K-го наименьшего храним **K** самых маленьких чисел в куче с отрицательными значениями (аналог макс-кучи). Для K-го наибольшего используем обычную мин-кучу размера K. Сложность: **O(n log K)** по времени и **O(K)** по памяти.

In [6]:
def kth_smallest(nums, k):
    if k < 1 or k > len(nums):
        raise ValueError('Некорректное k')

    heap = [-x for x in nums[:k]]
    heapq.heapify(heap)

    for x in nums[k:]:
        if x < -heap[0]:
            heapq.heapreplace(heap, -x)

    return -heap[0]


def kth_largest(nums, k):
    if k < 1 or k > len(nums):
        raise ValueError('Некорректное k')

    heap = nums[:k]
    heapq.heapify(heap)

    for x in nums[k:]:
        if x > heap[0]:
            heapq.heapreplace(heap, x)

    return heap[0]


numbers = [7, 2, 9, 4, 2, 10]
assert kth_smallest(numbers, 3) == 4
assert kth_smallest(numbers, 2) == 2
assert kth_largest(numbers, 2) == 9
assert kth_largest(numbers, 4) == 4
print('Задача 3:', kth_smallest(numbers, 3), kth_largest(numbers, 2))

Задача 3: 4 9


## 4. K-й элемент в BST
В BST симметричный обход (лево -> корень -> право) выдаёт значения по возрастанию. Для наибольшего элемента идём в обратном порядке. Сложность: **O(h + K)** по времени и **O(h)** по памяти, где h - высота дерева.

In [7]:
def kth_in_bst(root, k, largest=False):
    if k < 1:
        raise ValueError('Некорректное k')

    stack = []
    current = root
    count = 0

    while stack or current:
        while current:
            stack.append(current)
            current = current.right if largest else current.left

        current = stack.pop()
        count += 1
        if count == k:
            return current.value

        current = current.left if largest else current.right

    raise ValueError('k больше числа узлов')


bst = Node(5, Node(3, Node(2), Node(4)), Node(8, None, Node(10, Node(9))))

assert kth_in_bst(bst, 1) == 2
assert kth_in_bst(bst, 3) == 4
assert kth_in_bst(bst, 2, largest=True) == 9
assert kth_in_bst(bst, 1, largest=True) == 10
print('BST:')
print_tree(bst)
print('Задача 4:', kth_in_bst(bst, 3), kth_in_bst(bst, 2, largest=True))

BST:
                            5
            3                               8
    2               4                               10
                                                9
Задача 4: 4 9


## 5. Balance factor
Высота пустого дерева равна 0, высота листа - 1. Рекурсивно считаем высоты поддеревьев снизу вверх и записываем `balance_factor = высота_левого - высота_правого`. Сложность: **O(n)** по времени и **O(h)** по памяти из-за рекурсии.

In [8]:
def fill_balance_factors(root):
    if root is None:
        return 0

    left_height = fill_balance_factors(root.left)
    right_height = fill_balance_factors(root.right)

    root.balance_factor = left_height - right_height
    return max(left_height, right_height) + 1


height = fill_balance_factors(bst)
assert height == 4
assert bst.balance_factor == -1
assert bst.left.balance_factor == 0
assert bst.right.balance_factor == -2
assert bst.right.right.balance_factor == 1
print('Дерево с вычисленными факторами баланса:')
print_tree(bst)
print('Баланс узлов:')
q = deque([bst])
while q:
    node = q.popleft()
    print(node.value, '->', node.balance_factor)
    if node.left is not None:
        q.append(node.left)
    if node.right is not None:
        q.append(node.right)
print('Задача 5: высота =', height, ', баланс корня =', bst.balance_factor)

Дерево с вычисленными факторами баланса:
                            5
            3                               8
    2               4                               10
                                                9
Баланс узлов:
5 -> -1
3 -> 0
8 -> -2
2 -> 0
4 -> 0
10 -> 1
9 -> 0
Задача 5: высота = 4 , баланс корня = -1


## 6. Зеркальное дерево
В каждом узле меняем местами левое и правое поддеревья, затем зеркально отражаем оба. Так операция выполняется для всех узлов. Сложность: **O(n)** по времени и **O(h)** по памяти из-за рекурсии.

In [9]:
def mirror_tree(root):
    if root is None:
        return None

    root.left, root.right = root.right, root.left
    mirror_tree(root.left)
    mirror_tree(root.right)
    return root


def tree_as_tuple(root):
    if root is None:
        return None
    return (root.value, tree_as_tuple(root.left), tree_as_tuple(root.right))


before = tree_as_tuple(bst)
print('До отражения:')
print_tree(bst)
mirror_tree(bst)
print('После отражения:')
print_tree(bst)

assert bst.left.value == 8
assert bst.right.value == 3
assert bst.left.left.value == 10
assert bst.right.right.value == 2

mirror_tree(bst)
assert tree_as_tuple(bst) == before
assert mirror_tree(None) is None
print('После повторного отражения:')
print_tree(bst)
print('Задача 6: OK')

До отражения:
                            5
            3                               8
    2               4                               10
                                                9
После отражения:
                            5
            8                               3
    10                              4               2
        9
После повторного отражения:
                            5
            3                               8
    2               4                               10
                                                9
Задача 6: OK
